# 2. Extract resume text

The application form export stores each resume as a hosted PDF link. This notebook downloads every PDF, extracts the text with PyMuPDF, and writes `Extracted_Resumes.csv` (`Name`, `Resume_Text`).

In [ ]:
!pip install PyMuPDF

In [ ]:
import csv
import re
import requests
import fitz  # PyMuPDF
import os
import pandas as pd

def get_pdf_text_from_url(pdf_url, temp_filename="temp_resume.pdf"):
    """Download a PDF and return its plain text."""
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
        }
        # 1. download the PDF
        response = requests.get(pdf_url, headers=headers, stream=True, timeout=15)
        response.raise_for_status()

        with open(temp_filename, 'wb') as file:
            for chunk in response.iter_content(chunk_size=8192):
                file.write(chunk)

        # 2. extract text
        extracted_text = ""
        with fitz.open(temp_filename) as pdf_document:
            for page_num in range(len(pdf_document)):
                extracted_text += pdf_document[page_num].get_text()

        return extracted_text.strip()

    except Exception as e:
        print(f"  extraction failed: {e}")
        return "[EXTRACTION_FAILED]"

    finally:
        # remove the temporary file
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

def create_resume_text_csv(input_csv, output_csv="Extracted_Resumes.csv"):
    """Read the submission export (with resume links) and write Name + Resume_Text."""
    print(f"Reading {input_csv}...")

    results = []

    # utf-8-sig handles the BOM in the export
    with open(input_csv, mode='r', encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)

        for row_num, row in enumerate(reader, start=1):
            name = row.get('🧑\u200d🎓 Name', f'unknown_{row_num}')
            resume_data = row.get('🎯 Resume', '')

            # the resume field looks like 'file.pdf (https://...)'; pull out the URL
            match = re.search(r'\((https?://[^)]+)\)', resume_data)

            if match:
                pdf_url = match.group(1)
                print(f"[{row_num}] extracting resume...")

                
                text_content = get_pdf_text_from_url(pdf_url)

                
                results.append({
                    "Name": name,
                    "Resume_Text": text_content
                })
            else:
                print(f"[{row_num}] no PDF link found.")
                results.append({
                    "Name": name,
                    "Resume_Text": "[NO_LINK]"
                })

    # write results
    if results:
        df = pd.DataFrame(results)
        df.to_csv(output_csv, index=False, encoding='utf-8-sig')
        print(f"\nDone. Saved {output_csv}")
    else:
        print("\nNo rows extracted.")

# === run ===
if __name__ == "__main__":
    # submission export (same folder as this notebook)
    INPUT_CSV = "Submission.csv" 

    
    OUTPUT_CSV = "Extracted_Resumes.csv"

    create_resume_text_csv(INPUT_CSV, OUTPUT_CSV)